<a href="https://colab.research.google.com/github/4cekay/B101-Group2-NLP-Project/blob/main/DatasetCuration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Setup**

In [1]:
!pip install numpy datasets


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [41]:
import pandas as pd
import glob
import re
from datasets import load_dataset

**Helper #1:** cleaning formatted text (particularly for generated articles):

- Escaped strings
- Special tokens
- Formatting, markdown
- Metadata
- Placeholders
- Footers, promotional plugs

In [124]:
def clean_article(text, max_words=250):
  if not text:
    return ""

  # if applicable, handle ALL instances of the following:

  # Escaped Strings
  text = text.replace("\\n", "\n")

  # Special Token (i.e. gemma's <end_of_turn>)
  text = re.sub(r"<end_of_turn>", "", text)

  # Formatting
  text = re.sub(r"^\s*#+\s*", "", text, flags=re.MULTILINE)         # header hashes
  text = re.sub(r"\*\*(.*?)\*\*", r"\1", text)                      # bold asterisks
  text = re.sub(r"^\s[\*\-]\s*", "", text, flags=re.MULTILINE)      # bullet points
  text = re.sub(r"^\s*[-*_]{3,}\s*$", "", text, flags=re.MULTILINE) # dividers

  # Metadata
  text = re.sub(r"^\s*(?:#+|\*\*)*\s*title\s*|(?:\*\*)?\s*:\s*", "", text, flags=re.IGNORECASE | re.MULTILINE)      # "Title:"
  text = re.sub(r"^\s*(?:#+|\*\*)*\s*by\b.*$", "", text, flags=re.IGNORECASE | re.MULTILINE)                         # "by" writer
  text = re.sub(r"^\s(?:#+|\*\*)*\s*(?:published|date|author)\s*:.*$", "", text, flags=re.IGNORECASE | re.MULTILINE) # publication details

  # Placeholders
  text = re.sub(r"\[\s*(?:insert|your|author|date|link|chart|image)[^\]]*\]", "", text, flags=re.IGNORECASE)

  # Footers
  text = re.sub(r"^\s*(?:#+|\*\*)*\s*(?:published|date|author)\s*:.*$", "", text, flags=re.IGNORECASE | re.MULTILINE)

  # keep # of paragraphs sampled within the word count (split by \n\n)
  paragraphs = [p.strip() for p in re.split(r'\n\s*\n+', text) if p.strip()]
  kept_paragraphs = []
  total_word_count = 0

  for p in paragraphs: # check each paragraph to make sure sample stays within max
    para_word_count = len(p.split())

    if total_word_count + para_word_count <= max_words:
      kept_paragraphs.append(p)
      total_word_count += para_word_count
    else:
      break

  # If the first paragraph is already too long, just don't use the sample
  if not kept_paragraphs:
    return None

  return "\n\n".join(kept_paragraphs)

**Helper #2:** Gathering test/val split from a random subset of a chosen size

In [110]:
def split_subset(samples, subset_size, min_words, max_words, train_split=0.8, column="sample_text"):
  if samples is None or len(samples) ==0:
    return pd.DataFrame(), pd.DataFrame()

  # subset of random samples within set
  subset = (
      samples
      .filter(lambda example: min_words <= len(example[column].split()) <= max_words)
      .shuffle(seed=42)
      .select(range(subset_size))
  )

  # subset to dataframe + clear \n
  df_subset = subset.to_pandas()
  df_subset[column] =  df_subset[column].str.replace(r'[\r\n]+', " ", regex=True).str.strip()

  # split subset into train and val
  n_train = round(len(df_subset) * train_split) # number of training samples

  df_subset_train = df_subset.sample(n=n_train, random_state=42) # get train samples
  df_subset_val = df_subset.drop(df_subset_train.index)          # remove train samples to get val remainder


  return df_subset_train.reset_index(drop=True), df_subset_val.reset_index(drop=True)

# **Extraction**
[HF Dataset Processing](https://huggingface.co/docs/datasets/v1.4.0/processing.html)


## Email Replies

### Human-written

**Stanford Humanual-Email Dataset**

Columns:
- **completion:** ground-truth email reply
  - Main target text sample !!
- post_id (discard)
- user_id (discard)
- timestamp (discard)
- turn_id (discard)
  - Note: In conversation "turn 1" rows, I can take the original email content to use as a prompt for AI-generated samples to compare (w/ proper attribution)
- persona (discard)
- **prompt:** role and content
  - Will pull the initial email content of select conversations to use in prompting later.  
- metadata (discard)


In [3]:
# Load the raw dataset (https://huggingface.co/datasets/snap-stanford/humanual-email)

raw_dataset = load_dataset('snap-stanford/humanual-email')
raw_dataset


README.md:   0%|          | 0.00/2.71k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 15.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  675kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/val-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/6377 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/536 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/130 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['completion', 'post_id', 'user_id', 'timestamp', 'turn_id', 'persona', 'prompt', 'metadata'],
        num_rows: 6377
    })
    test: Dataset({
        features: ['completion', 'post_id', 'user_id', 'timestamp', 'turn_id', 'persona', 'prompt', 'metadata'],
        num_rows: 536
    })
    val: Dataset({
        features: ['completion', 'post_id', 'user_id', 'timestamp', 'turn_id', 'persona', 'prompt', 'metadata'],
        num_rows: 130
    })
})

1. I'm gonna extract the initial email content of each conversation thread to use as prompts for our generative email samples later.

In [4]:
# extracting only the rows with turn_id == 1, indicating the first "turn" in a round of emails

initial_emails = raw_dataset.filter(lambda example: example["turn_id"] == 1)

Filter:   0%|          | 0/6377 [00:00<?, ? examples/s]

Filter:   0%|          | 0/536 [00:00<?, ? examples/s]

Filter:   0%|          | 0/130 [00:00<?, ? examples/s]

In [5]:
# removing all unwanted features, keeping only the "completion" and "prompt"
initial_emails = initial_emails.remove_columns(["post_id", "user_id", "timestamp", "turn_id", "persona", "metadata"])

2. Now, we take the actual email replies to use as training data.

In [6]:
email_replies = initial_emails.map(
    lambda example: {
        "sample_replies": example["completion"].strip('\'"')
    },
    remove_columns= ["prompt", "completion"]
)

Map:   0%|          | 0/3780 [00:00<?, ? examples/s]

Map:   0%|          | 0/371 [00:00<?, ? examples/s]

Map:   0%|          | 0/96 [00:00<?, ? examples/s]

3. Take 100 samples from the train set that are within the word count range.

In [19]:
max_words = 250
min_words = 20
sample_count = 100

email_replies_subset = (
    email_replies["train"]
    .filter(lambda example: min_words <= len(example["sample_replies"].split()) <= max_words)
    .shuffle(seed=42) # maintain same samples in notebook
    .select(range(sample_count))
)

Filter:   0%|          | 0/3780 [00:00<?, ? examples/s]

In [20]:
df_email_replies = email_replies_subset.to_pandas()
print(len(email_replies_subset))

df_email_replies

100


,sample_replies
0,"Louise,\n\nWe are happy to do this.\n\nShall w..."
1,"Per Pam, the person we should be talking to is..."
2,"Louise,\n\nAttached is the list of Top 50 coun..."
3,Sounds great. Just check with Audrey (3-5849) ...
4,Sally\n\nProposed list - of all potential cand...
...,...
95,"As requested below, please now use Exelon Gene..."
96,Hi Bruce/Edward - \n\nI am preparing the Termi...
97,Wait until Louise gets back. These should not...
98,Get the specifics we are organizing a meeting ...


In [21]:
# index=False to get rid of the index no. column

df_email_replies.to_csv('human_email_samples.csv', index=False)

### AI-generated

Gathering the context (prompt --> content) for a number of email replies (completion) in the dataset.
These will be fed into different generative AI models, asking for an appropriate response.

In [23]:
# extract the email content from each prompt,
# get rid of quotation marks from the beginning and end
# remove the old columns
email_prompts = initial_emails.map(
    lambda example: {
        "text": example["prompt"][0]["content"].strip('\'"') # get rid of quotes
    },
    remove_columns= ["completion", "prompt"]
)

Map:   0%|          | 0/3780 [00:00<?, ? examples/s]

Map:   0%|          | 0/371 [00:00<?, ? examples/s]

Map:   0%|          | 0/96 [00:00<?, ? examples/s]

In [26]:
max_words = 250
min_words = 20
sample_count = 25

email_prompts_subset = (
    email_prompts["train"]
    .filter(lambda example: min_words <= len(example["text"].split()) <= max_words)
    .shuffle(seed=42) # maintain same samples in notebook
    .select(range(sample_count))
)

In [28]:
df_email_prompts = email_prompts_subset.to_pandas()

df_email_prompts

,text
0,The Ninth Circuit Court of Appeals has schedul...
1,Ken Lay spoke with several California CEOs thi...
2,"TJ,\n\nThe following Realtime employees on the..."
3,Revised Trading Policy. We would like to get ...
4,Legal -\n\nAttached below is an Omnibus Confir...
5,I have 4 tickets for the Astros game on Thursd...
6,Any more progress on the Management Conference...
7,Kevin:\n\nPer you request I left a phone messa...
8,Before you leave today or first thing in the m...
9,"Tara and Melba,\n\nTori Kuykendahl would like ..."


In [29]:
# csv export example (index=False to get rid of the index no. column)

df_email_prompts.to_csv('human_email_prompts.csv', index=False)

AI-Generated Replies to each of those 25 prompt emails:

Prompt: "Can you reply (within 250 words) to each email in the attached file in an appropriate, professional manner?" Export draft replies into csv.

[Attached human_email_prompts.csv]

In [8]:
path = r"C:\Users\kacey\OneDrive\IAT360\NLP_Project\generated_samples\*.csv"
ai_samples_csv = glob.glob(path)

ai_email_train = []
ai_email_val = []

for file in ai_samples_csv:
  df = pd.read_csv(file)

  # 5 random rows
  val_sample = df.sample(n=5, random_state=42)
  ai_email_val.append(val_sample)

  # rest to training
  train_sample = df.drop(val_sample.index)
  ai_email_train.append(train_sample)

# concat
df_ai_email_train = pd.concat(ai_email_train, ignore_index=True)
df_ai_email_val = pd.concat(ai_email_val, ignore_index=True)

# save to their own csv file
df_ai_email_train.to_csv('train_ai_email_samples.csv', index=False)
df_ai_email_val.to_csv('val_ai_email_samples.csv', index=False)

print(f"Validation rows: {len(df_ai_email_val)}")
print(f"Training rows: {len(df_ai_email_train)}")


Validation rows: 20
Training rows: 80


## Abstracts

### AI-generated

In [9]:
# https://huggingface.co/datasets/Ateeqq/AI-and-Human-Generated-Text

raw_dataset = load_dataset('Ateeqq/AI-and-Human-Generated-Text')
raw_dataset


README.md:   0%|          | 0.00/1.46k [00:00<?, ?B/s]

train.csv: reconstructing file:   0%|          |  0.00B / 33.9MB            

train.csv: downloading bytes:           |  0.00B            

test.csv:   0%|          | 0.00/8.74M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/22930 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5732 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['title', 'abstract', 'label'],
        num_rows: 22930
    })
    test: Dataset({
        features: ['title', 'abstract', 'label'],
        num_rows: 5732
    })
})

In [10]:
# extracting only the AI-generated samples (label = 1)

initial_abstracts = raw_dataset.filter(lambda example: example["label"] == 1)

# removing all unwanted features, keeping only "abstract"
initial_abstracts = initial_abstracts.remove_columns(["title", "label"])

initial_abstracts

Filter:   0%|          | 0/22930 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5732 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['abstract'],
        num_rows: 11465
    })
    test: Dataset({
        features: ['abstract'],
        num_rows: 2866
    })
})

In [24]:
max_words = 250
sample_count = 100

abstracts_subset = (
    initial_abstracts["train"]
    .filter(lambda example: len(example["abstract"].split()) <= max_words)
    .shuffle(seed=42) # maintain same samples in notebook
    .select(range(sample_count))
)

In [91]:
df_abstracts = abstracts_subset.to_pandas()
df_abstracts =  df_abstracts.replace(r'\n+|\r+', ' ', regex=True)

print(len(abstracts_subset))

df_abstracts.to_csv('ai_abstracts.csv', index=False)

100


In [92]:
df_abstracts

,abstract
0,This article presents a systematic review of ...
1,This paper introduces the practical steps for...
2,This study explores the novel use of isotherm...
3,This study explores the mechanism by which he...
4,This article explores the intersection of the...
...,...
95,"This study focuses on the incidence, predict..."
96,This paper describes the successful developme...
97,This research article investigates the potent...
98,This paper explores the potential of in vitro...


In [32]:
ai_abstract_train = []
ai_abstract_val = []

val_sample = df_abstracts.sample(n=20, random_state=42)
ai_abstract_val.append(val_sample)

train_sample = df_abstracts.drop(val_sample.index)
ai_abstract_train.append(train_sample)

# concat into dataframes for each split
df_ai_abstract_train = pd.concat(ai_abstract_train, ignore_index=True)
df_ai_abstract_val = pd.concat(ai_abstract_val, ignore_index=True)

# save to their own csv file
df_ai_abstract_train.to_csv('train_ai_abstract_samples.csv', index=False)
df_ai_abstract_val.to_csv('val_ai_abstract_samples.csv', index=False)

print(f"Validation rows: {len(df_ai_abstract_val)}")
print(f"Training rows: {len(df_ai_abstract_train)}")

Validation rows: 20
Training rows: 80


In [33]:
df_ai_abstract_train

,abstract
0,This paper introduces the practical steps for...
1,This study explores the novel use of isotherm...
2,This study explores the mechanism by which he...
3,This paper examines the effectiveness of part...
4,Airborne/Droplet Infection Isolation: A Compr...
...,...
75,"This study focuses on the incidence, predict..."
76,This paper describes the successful developme...
77,This research article investigates the potent...
78,This paper explores the potential of in vitro...


## News Articles

### AI-generated

In [36]:
# https://huggingface.co/datasets/gsingh1-py/train

raw_dataset = load_dataset('gsingh1-py/train')
raw_dataset

Repo card metadata block was not found. Setting CardData to empty.


train.csv: reconstructing file:   0%|          |  0.00B /  161MB            

train.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7321 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'Human_story', 'gemma-2-9b', 'mistral-7B', 'qwen-2-72B', 'llama-8B', 'accounts/yi-01-ai/models/yi-large', 'GPT_4-o'],
        num_rows: 7321
    })
})

In [125]:
# extracting and cleaning from each model's samples

model_columns = ["gemma-2-9b", "qwen-2-72B", "llama-8B", "GPT_4-o" ]

cleaned_datasets = {}

for model in model_columns:
  if model in raw_dataset["train"].column_names:
    cleaned_datasets[model] = (
        raw_dataset.map(
            lambda example: {"sample_text": clean_article(example[model])},
            remove_columns=raw_dataset["train"].column_names
        ) # filter out the blanks (cleared out after not fitting in word count)
        .filter(lambda example: example["sample_text"] is not None and len(example["sample_text"].strip()) > 0)
    )

Map:   0%|          | 0/7321 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7321 [00:00<?, ? examples/s]

Map:   0%|          | 0/7321 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7321 [00:00<?, ? examples/s]

Map:   0%|          | 0/7321 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7321 [00:00<?, ? examples/s]

Map:   0%|          | 0/7321 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7321 [00:00<?, ? examples/s]

In [126]:
cleaned_datasets

gemma_articles = cleaned_datasets['gemma-2-9b']['train']
qwen_articles = cleaned_datasets['qwen-2-72B']['train']
llama_articles = cleaned_datasets['llama-8B']['train']
gpt_articles = cleaned_datasets['GPT_4-o']['train']

In [127]:
gemma_train_split, gemma_val_split = split_subset(gemma_articles, 25, 50, 250)
qwen_train_split, qwen_val_split = split_subset(qwen_articles, 25, 50, 250)
llama_train_split, llama_val_split = split_subset(llama_articles, 25, 50, 250)
gpt_train_split, gpt_val_split = split_subset(gpt_articles, 25, 50, 250)

Filter:   0%|          | 0/7310 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5683 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7306 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7285 [00:00<?, ? examples/s]

In [128]:
ai_articles_train = [gemma_train_split, qwen_train_split, llama_train_split, gpt_train_split]
ai_articles_val = [gemma_val_split, qwen_val_split, llama_val_split, gpt_val_split]

# concat
df_ai_articles_train = pd.concat(ai_articles_train, ignore_index=True)
df_ai_articles_val = pd.concat(ai_articles_val, ignore_index=True)

# save to their own csv file
df_ai_articles_train.to_csv('train_ai_articles_samples.csv', index=False)
df_ai_articles_val.to_csv('val_ai_articles_samples.csv', index=False)

print(f"Validation rows: {len(df_ai_articles_val)}")
print(f"Training rows: {len(df_ai_articles_train)}")

Validation rows: 20
Training rows: 80


# **Compilation**